# MMS-300M Fast Fallback Evaluation (Kaggle)
Uses HF training artifacts + Inference API to get metrics quickly when local model loading fails.

In [ ]:
!pip -q install huggingface_hub pandas numpy scikit-learn seaborn matplotlib librosa soundfile

import os, json, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from huggingface_hub import HfApi, hf_hub_download, InferenceClient

warnings.filterwarnings("ignore")
random.seed(42)
np.random.seed(42)

# -----------------------------
# CONFIG
# -----------------------------
MODEL_ID = "Raemih/mms-multilingual-ser"
HF_TOKEN = os.environ.get("HF_TOKEN", None)  # set Kaggle secret
OUT_DIR = Path("/kaggle/working/mms300m_fast_eval")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_TEST_PER_LANG = 100
EMOTIONS = ["neutral", "happy", "sad", "angry", "fear"]

# dataset paths
RAVDESS_PATH = "/kaggle/input/datasets/uwrfkaggler/ravdess-emotional-speech-audio"
TESS_PATH    = "/kaggle/input/datasets/ejlok1/toronto-emotional-speech-set-tess"
TAMIL_PATH   = "/kaggle/input/datasets/senadhithimanya/emo-ta"
SINHALA_PATH = "/kaggle/input/datasets/senadhithimanya/sinhala-labeled-100"

AUDIO_EXTS = {".wav", ".mp3", ".flac", ".ogg", ".m4a"}

def to_path(p):
    return Path(p) if p is not None else None

def find_audio_files(base_path):
    base_path = to_path(base_path)
    if base_path is None or not base_path.exists():
        return []
    return [str(p) for p in base_path.rglob("*") if p.is_file() and p.suffix.lower() in AUDIO_EXTS]

def detect_emotion_from_text(s):
    s = str(s).lower()
    for e in EMOTIONS:
        if e in s:
            return e
    if "_ps" in s or "ps_" in s:
        return "happy"
    if "anger" in s:
        return "angry"
    if "fearful" in s:
        return "fear"
    return None

def prepare_ravdess(path):
    rows = []
    emo_map = {"01":"neutral","03":"happy","04":"sad","05":"angry","06":"fear"}
    for fp in find_audio_files(path):
        parts = Path(fp).stem.split("-")
        if len(parts) >= 3 and parts[2] in emo_map:
            rows.append({"path": fp, "emotion": emo_map[parts[2]], "language":"english"})
    return pd.DataFrame(rows)

def prepare_tess(path):
    rows = []
    for fp in find_audio_files(path):
        emo = detect_emotion_from_text(fp)
        if emo in EMOTIONS:
            rows.append({"path": fp, "emotion": emo, "language":"english"})
    return pd.DataFrame(rows)

def prepare_tamil(path):
    rows = []
    for fp in find_audio_files(path):
        emo = detect_emotion_from_text(fp)
        if emo in EMOTIONS:
            rows.append({"path": fp, "emotion": emo, "language":"tamil"})
    return pd.DataFrame(rows)

def prepare_sinhala(path):
    path = to_path(path)
    rows = []
    if path is None or not path.exists():
        return pd.DataFrame(columns=["path","emotion","language"])

    csvs = list(path.rglob("*.csv"))
    for c in csvs:
        try:
            df = pd.read_csv(c)
        except:
            continue
        cols = {x.lower(): x for x in df.columns}
        pcol = next((cols[k] for k in ["path","filepath","file_path","audio","audio_path","filename","file"] if k in cols), None)
        ecol = next((cols[k] for k in ["emotion","label","class","target"] if k in cols), None)
        if pcol and ecol:
            for _, r in df.iterrows():
                emo = detect_emotion_from_text(r[ecol])
                rp = Path(str(r[pcol]))
                if not rp.is_absolute():
                    rp = (c.parent / rp).resolve()
                if emo in EMOTIONS and rp.exists() and rp.suffix.lower() in AUDIO_EXTS:
                    rows.append({"path": str(rp), "emotion": emo, "language":"sinhala"})
            if rows:
                return pd.DataFrame(rows)

    for fp in find_audio_files(path):
        emo = detect_emotion_from_text(fp)
        if emo in EMOTIONS:
            rows.append({"path": fp, "emotion": emo, "language":"sinhala"})
    return pd.DataFrame(rows)

def stratified_test_split(df, test_size=0.15, seed=42):
    if len(df) < 10:
        return df.copy().reset_index(drop=True)
    _, test_df = train_test_split(df, test_size=test_size, random_state=seed, stratify=df["emotion"])
    return test_df.reset_index(drop=True)

def cap_df(df, n):
    if len(df) <= n:
        return df.reset_index(drop=True)
    return df.sample(n=n, random_state=42).reset_index(drop=True)

# -----------------------------
# 1) Pull train/val metrics from HF artifacts
# -----------------------------
api = HfApi(token=HF_TOKEN)
files = api.list_repo_files(MODEL_ID)

train_metrics = {
    "train_loss": None,
    "train_accuracy": None,
    "validation_loss": None,
    "validation_accuracy": None,
    "epochs_run": None
}

all_results_candidates = [f for f in files if f.endswith("all_results.json")]
if all_results_candidates:
    p = hf_hub_download(MODEL_ID, all_results_candidates[0], token=HF_TOKEN)
    d = json.load(open(p, "r"))
    train_metrics["train_loss"] = d.get("train_loss")
    train_metrics["validation_loss"] = d.get("eval_loss")
    train_metrics["validation_accuracy"] = d.get("eval_accuracy")
    train_metrics["train_accuracy"] = d.get("train_accuracy")
    train_metrics["epochs_run"] = d.get("epoch")

state_candidates = [f for f in files if f.endswith("trainer_state.json")]
if state_candidates:
    p = hf_hub_download(MODEL_ID, state_candidates[0], token=HF_TOKEN)
    st = json.load(open(p, "r"))
    logs = st.get("log_history", [])
    last_train = None
    last_eval = None
    for x in logs:
        if "loss" in x and "eval_loss" not in x:
            last_train = x
        if "eval_loss" in x:
            last_eval = x
    if last_train:
        train_metrics["train_loss"] = train_metrics["train_loss"] or last_train.get("loss")
        train_metrics["train_accuracy"] = train_metrics["train_accuracy"] or last_train.get("accuracy")
        train_metrics["epochs_run"] = train_metrics["epochs_run"] or last_train.get("epoch")
    if last_eval:
        train_metrics["validation_loss"] = train_metrics["validation_loss"] or last_eval.get("eval_loss")
        train_metrics["validation_accuracy"] = train_metrics["validation_accuracy"] or last_eval.get("eval_accuracy")
        train_metrics["epochs_run"] = train_metrics["epochs_run"] or last_eval.get("epoch")

if train_metrics["epochs_run"] is None:
    train_metrics["epochs_run"] = 25  # commit config fallback

print("Train/Val metrics from HF artifacts:", train_metrics)

# -----------------------------
# 2) Build capped test sets
# -----------------------------
eng = pd.concat([prepare_ravdess(RAVDESS_PATH), prepare_tess(TESS_PATH)], ignore_index=True)
tam = prepare_tamil(TAMIL_PATH)
sin = prepare_sinhala(SINHALA_PATH)

for name, df in [("english", eng), ("tamil", tam), ("sinhala", sin)]:
    if len(df) == 0:
        raise RuntimeError(f"No data found for {name}")

eng_test = cap_df(stratified_test_split(eng), MAX_TEST_PER_LANG)
tam_test = cap_df(stratified_test_split(tam), MAX_TEST_PER_LANG)
sin_test = cap_df(stratified_test_split(sin), MAX_TEST_PER_LANG)

tests = {"english": eng_test, "tamil": tam_test, "sinhala": sin_test}
print({k: len(v) for k,v in tests.items()})

# -----------------------------
# 3) HF inference API eval
# -----------------------------
client = InferenceClient(token=HF_TOKEN)

def norm_pred_label(lbl):
    s = str(lbl).lower()
    if "anger" in s: return "angry"
    if "fear" in s: return "fear"
    if "happy" in s: return "happy"
    if "neutral" in s: return "neutral"
    if "sad" in s: return "sad"
    return None

def predict_label(audio_path):
    out = client.audio_classification(audio=audio_path, model=MODEL_ID)
    best_label, best_score = None, -1
    for x in out:
        lbl = norm_pred_label(x.get("label"))
        sc = float(x.get("score", 0))
        if lbl in EMOTIONS and sc > best_score:
            best_label, best_score = lbl, sc
    return best_label

lang_results = []
for lang, df in tests.items():
    y_true, y_pred = [], []
    for _, r in df.iterrows():
        y_true.append(r["emotion"])
        try:
            p = predict_label(r["path"])
        except Exception:
            p = None
        if p is None:
            p = "neutral"
        y_pred.append(p)

    acc = accuracy_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred, labels=EMOTIONS)
    cr_txt = classification_report(y_true, y_pred, labels=EMOTIONS, digits=4)
    cr_json = classification_report(y_true, y_pred, labels=EMOTIONS, output_dict=True)

    d = OUT_DIR / lang
    d.mkdir(parents=True, exist_ok=True)
    np.save(d / "confusion_matrix.npy", cm)
    with open(d / "classification_report.txt", "w") as f: f.write(cr_txt)
    with open(d / "classification_report.json", "w") as f: json.dump(cr_json, f, indent=2)

    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=EMOTIONS, yticklabels=EMOTIONS)
    plt.title(f"Confusion Matrix - {lang}")
    plt.tight_layout()
    plt.savefig(d / "confusion_matrix.png", dpi=200)
    plt.close()

    lang_results.append({
        "language": lang,
        "train_accuracy": train_metrics["train_accuracy"],
        "train_loss": train_metrics["train_loss"],
        "validation_accuracy": train_metrics["validation_accuracy"],
        "validation_loss": train_metrics["validation_loss"],
        "epochs_run": train_metrics["epochs_run"],
        "test_accuracy": acc
    })

# -----------------------------
# 4) Final table
# -----------------------------
final_df = pd.DataFrame(lang_results)
final_df.to_csv(OUT_DIR / "mms300m_fast_table.csv", index=False)

print("\nFinal table:")
display(final_df)
print(f"\nSaved outputs in: {OUT_DIR}")
